# Refill + ConvLSTM on the dredging page: publish and verify

Runs on a normal Colab CPU runtime. It does two things:

1. **Publish**: runs our model on the same 6 historical frames as the XGBoost and physics models and writes the
   files the page reads (`predictions/convlstm/*.geojson`, plus new entries in `manifest.json` and `comparisons.json`).
2. **Verify**: starts the real Flask server and checks the historical files and the live `/api/dredging/forecast` route.

At the end it downloads `dredging_convlstm_outputs.zip`, which goes into `backend/static/data/anchorage/`.

## 1. Upload the backend and install

In [ ]:
import os
if not os.path.exists('backend'):
    from google.colab import files
    files.upload()                                   # backend_colab.zip
    !unzip -q -o backend_colab.zip
!pip install -q -r backend/requirements.txt

## 2. Precompute our model for the page's 6 frames

Each frame is a fixed pair: origin survey → 28 days later. We force our model to use exactly that origin survey,
and keep only the cells that the other two models also show, so all three are scored on the same cells.

In [ ]:
import hashlib, json, sys
from pathlib import Path
import numpy as np
from pyproj import Transformer

os.environ.setdefault('SECRET_KEY', 'colab-only-not-secret')   # importing `app` loads the website config, which requires it
sys.path.insert(0, 'backend')
from app import dredging_forecast as df

DATA = Path('backend/static/data/anchorage')
OUT = DATA / 'predictions' / 'convlstm'
OUT.mkdir(parents=True, exist_ok=True)
manifest = json.loads((DATA / 'manifest.json').read_text())
comparisons = json.loads((DATA / 'comparisons.json').read_text())
bundle = df.load_bundle()
G = bundle['grid']
H, W = G['shape']
to_plane = Transformer.from_crs('EPSG:4326', 'EPSG:26934', always_xy=True)
usable = df.load_catalog()

def cells_of(geojson_path):
    """Grid mask of the cells in one of the other models' files."""
    mask = np.zeros((H, W), bool)
    for f in json.loads(Path(geojson_path).read_text())['features']:
        ring = np.asarray(f['geometry']['coordinates'][0][:4])
        x, y = to_plane.transform(ring[:, 0], ring[:, 1])
        i, j = round(y.min() / G['cell_m']) - G['row0'], round(x.min() / G['cell_m']) - G['col0']
        if 0 <= i < H and 0 <= j < W:
            mask[i, j] = True
    return mask

def survey_on(day):
    rows = usable[usable.date == day]
    assert len(rows), f'no eHydro survey on {day}'
    return df.survey_map(rows.url.iloc[-1])

frame_comparisons = {c['id']: c for c in comparisons['frames']}
for frame in manifest['frames']:
    depth, change, show, props = df.forecast_arrays(frame['predicted_for'], origin=frame['from_survey'])
    assert props['from_survey'] == frame['from_survey'] and props['lead_days'] == 28
    theirs = cells_of('backend' + frame['outputs']['xgboost'])
    missing = (theirs & ~show).sum()
    show = show & theirs
    geojson = df.to_geojson(depth, change, show, bundle,
                            {'from_survey': frame['from_survey'], 'predicted_for': frame['predicted_for']})
    raw = json.dumps(geojson, separators=(',', ':'), allow_nan=False).encode()
    url = f"/static/data/anchorage/predictions/convlstm/convlstm-{frame['from_survey']}.geojson"
    Path('backend' + url).write_bytes(raw)
    frame['outputs']['convlstm'] = url

    origin_map, target_map = survey_on(frame['from_survey']), survey_on(frame['predicted_for'])
    pairs = [[round(float(origin_map[i, j]), 5), round(float(target_map[i, j]), 5)]
             if np.isfinite(origin_map[i, j]) and np.isfinite(target_map[i, j]) else None
             for i, j in zip(*np.where(show))]
    frame_comparisons[frame['id']]['outputs']['convlstm'] = {
        'forecast_url': url, 'forecast_sha256': hashlib.sha256(raw).hexdigest(), 'origin_target_depths_ft': pairs}
    print(frame['id'], len(geojson['features']), 'cells', f'({missing} of their cells we cannot predict)')

print('bundle scores:', bundle['scores'])
key = next(k for k in (bundle['model'], 'ensemble') if k in bundle['scores'])   # 'ensemble' = the 3-model average
score = bundle['scores'][key]['test']
manifest['models'] = [m for m in manifest['models'] if m['id'] != 'convlstm'] + [
    {'id': 'convlstm', 'label': 'Refill + ConvLSTM (also live)', 'mae_ft': float(score)}]
(DATA / 'manifest.json').write_text(json.dumps(manifest, separators=(',', ':')))
(DATA / 'comparisons.json').write_text(json.dumps(comparisons, separators=(',', ':'), allow_nan=False))
print('manifest models:', [m['id'] for m in manifest['models']], '| our test MAE', round(score, 3))

## 3. Same-cell accuracy on each frame

Mean absolute error of the predicted depth against the later survey, on the cells every model shows and both surveys
measured (the same numbers the page shows under "How close was this forecast?").

In [ ]:
import pandas as pd
rows = []
for frame in comparisons['frames']:
    outs = frame['outputs']
    files = {m: json.loads(Path('backend' + o['forecast_url']).read_text())['features'] for m, o in outs.items()}
    row = {'frame': frame['id']}
    for m, o in outs.items():
        err = [abs(f['properties']['depth_ft'] - p[1]) for f, p in zip(files[m], o['origin_target_depths_ft']) if p]
        base = [abs(p[1] - p[0]) for p in o['origin_target_depths_ft'] if p]
        row[m] = np.mean(err)
    row['no change'] = np.mean(base)
    rows.append(row)
table = pd.DataFrame(rows).set_index('frame')
table.loc['mean'] = table.mean()
table.round(3)

## 4. Start the real server

Needs `GOOGLE_MAPS_API_KEY` in Colab's 🔑 Secrets panel (only for the map in step 6). `SECRET_KEY` is a throwaway.

In [ ]:
import secrets, subprocess, time, requests
PORT = 8050
subprocess.run(['pkill', '-f', 'gunicorn'])
time.sleep(1)
env = dict(os.environ, SECRET_KEY=secrets.token_hex(32))
try:
    from google.colab import userdata
    env['GOOGLE_MAPS_API_KEY'] = userdata.get('GOOGLE_MAPS_API_KEY')
except Exception as e:
    print('no Maps key (the map will not load, everything else works):', e)
server = subprocess.Popen(['gunicorn', '--bind', f':{PORT}', '--workers', '1', '--threads', '4', '--timeout', '0', 'run:app'],
                          cwd='backend', env=env, stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(90):
    if server.poll() is not None:
        print('server crashed'); break
    try:
        requests.get(f'http://localhost:{PORT}/api/health', timeout=10); print('server is up'); break
    except requests.RequestException:
        time.sleep(2)
print(open('server.log').read()[-1500:])

## 5. Checks

In [ ]:
BASE = f'http://localhost:{PORT}'
ok = True
def check(name, condition, detail=''):
    global ok
    ok &= bool(condition)
    print('PASS' if condition else 'FAIL', name, detail)

# the page and its files are served
check('dredging page', requests.get(f'{BASE}/dredging').status_code == 200)
m = requests.get(f'{BASE}/static/data/anchorage/manifest.json').json()
check('manifest lists convlstm', any(x['id'] == 'convlstm' for x in m['models']))
for frame in m['frames']:
    r = requests.get(BASE + frame['outputs']['convlstm'])
    g = r.json() if r.ok else {}
    good = (r.ok and g.get('properties') == {'from_survey': frame['from_survey'], 'predicted_for': frame['predicted_for']}
            and all(len(f['geometry']['coordinates'][0]) == 5 and f['geometry']['coordinates'][0][0] == f['geometry']['coordinates'][0][4]
                    and np.isfinite(f['properties']['depth_ft']) and np.isfinite(f['properties']['change_ft']) for f in g['features']))
    check(f"historical file {frame['id']}", good, f"{len(g.get('features', []))} cells")

# live route: a real forecast, the cache, and the error cases
t = time.time(); r = requests.get(f'{BASE}/api/dredging/forecast', params={'date': '2026-11-13'}); first = time.time() - t
check('live forecast 2026-11-13', r.ok and len(r.json()['features']) > 0,
      f"{r.status_code}, {first:.0f} s, {r.json().get('properties') if r.ok else r.text[:200]}")
t = time.time(); r2 = requests.get(f'{BASE}/api/dredging/forecast', params={'date': '2026-11-13'})
check('second request is cached and identical', r2.ok and r2.content == r.content, f'{time.time() - t:.1f} s')
r = requests.get(f'{BASE}/api/dredging/forecast', params={'date': '2026-02-01'})
check('winter date is refused with a message', r.status_code == 400, r.json().get('error', ''))
r = requests.get(f'{BASE}/api/dredging/forecast', params={'date': 'banana'})
check('bad date is refused', r.status_code == 400)
print('\nALL CHECKS PASSED' if ok else '\nSOME CHECKS FAILED: see above, and !tail -50 server.log')

## 6. Look at the page (optional)

Shows the page inside the notebook. Pick **Refill + ConvLSTM** in the model list, move the slider, then choose
**Any date (live forecast)…** at the end of the date list. The satellite map may stay grey if the Maps key only allows
the real website; the forecast panel works either way.

In [ ]:
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(PORT, path='/dredging', height=900)

## 7. Download the published files

In [ ]:
import zipfile
with zipfile.ZipFile('dredging_convlstm_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in [DATA / 'manifest.json', DATA / 'comparisons.json', *sorted(OUT.glob('*.geojson'))]:
        z.write(path, path.relative_to(DATA))           # paths inside: manifest.json, predictions/convlstm/...
print(zipfile.ZipFile('dredging_convlstm_outputs.zip').namelist())
from google.colab import files
files.download('dredging_convlstm_outputs.zip')